In [3]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    classification_report,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.dummy import DummyClassifier

from xgboost import XGBClassifier
import joblib

# Load dataset

dataset = pd.read_csv("PhiUSIIL_Phishing_URL_Dataset.csv")

numeric_cols = [
    'URLLength',
    'DomainLength',
    'IsDomainIP',
    'TLDLength',
    'NoOfSubDomain',
    'IsHTTPS',
    'CharContinuationRate',
    'LetterRatioInURL',
    'DegitRatioInURL',
    'NoOfLettersInURL',
    'NoOfDegitsInURL',
    'NoOfEqualsInURL',
    'NoOfQMarkInURL',
    'NoOfAmpersandInURL',
    'NoOfOtherSpecialCharsInURL',
    'SpacialCharRatioInURL'
]

dominant_features = [
    'IsHTTPS',
    'NoOfOtherSpecialCharsInURL'
]

near_zero_features = [
    'NoOfQMarkInURL',
    'NoOfEqualsInURL',
    'NoOfAmpersandInURL',
    'IsDomainIP'
]

numeric_cols = [c for c in numeric_cols if c not in dominant_features]
numeric_cols = [c for c in numeric_cols if c not in near_zero_features]
numeric_cols = [c for c in numeric_cols if c in dataset.columns]

print("Features used:", numeric_cols)

# Check for possible leakage

print("Duplicate URLs:", dataset["URL"].duplicated().sum())

conflicting = dataset.groupby("URL")["label"].nunique()
print("URLs with conflicting labels:", (conflicting > 1).sum())

# Prepare data

X = dataset[numeric_cols]
y = dataset["label"]
groups = dataset["Domain"]

# Build pipeline

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols)
])

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ))
])

# Domain-isolated train/test split

gkf = GroupKFold(n_splits=5)

train_idx, test_idx = next(gkf.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

# Baseline

baseline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent"))
])

baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

print("\n--- Baseline ---")
print("Accuracy: {:.2f}%".format(
    accuracy_score(y_test, baseline_pred) * 100
))

# Test results

print("\n--- XGBoost Results ---")

cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix")
print(cm)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

tn, fp, fn, tp = cm.ravel()

false_positive_rate = fp / (fp + tn)
false_negative_rate = fn / (fn + tp)

print("\nAccuracy: {:.2f}%".format(
    accuracy * 100
))

print("\nClassification Report")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Phishing", "Legitimate"]
))

print("\nPerformance Summary")
print(f"Precision:            {precision:.4f}")
print(f"Recall:               {recall:.4f}")
print(f"F1 Score:             {f1:.4f}")
print(f"ROC-AUC:              {roc_auc:.4f}")
print(f"False Positive Rate:  {false_positive_rate:.2%}")
print(f"False Negative Rate:  {false_negative_rate:.2%}")

# Group cross validation

cv_scores = cross_val_score(
    pipeline,
    X,
    y,
    groups=groups,
    cv=gkf,
    scoring="accuracy"
)

print("\n--- GroupKFold Cross Validation ---")
print("Mean Accuracy: {:.2f}%".format(cv_scores.mean() * 100))
print("Std Dev: {:.2f}%".format(cv_scores.std() * 100))

# Feature importance

pipeline.fit(X, y)

importance = pd.Series(
    pipeline.named_steps["classifier"].feature_importances_,
    index=numeric_cols
).sort_values(ascending=False)

print("\n--- Feature Importance ---")
print(importance)

# Save trained model

#joblib.dump(pipeline, "phishing_detector.pkl")
#print("\nModel saved as phishing_detector.pkl")

Features used: ['URLLength', 'DomainLength', 'TLDLength', 'NoOfSubDomain', 'CharContinuationRate', 'LetterRatioInURL', 'DegitRatioInURL', 'NoOfLettersInURL', 'NoOfDegitsInURL', 'SpacialCharRatioInURL']
Duplicate URLs: 425
URLs with conflicting labels: 0

--- Baseline ---
Accuracy: 57.34%

--- XGBoost Results ---

Confusion Matrix
[[19997   119]
 [   27 27016]]

Accuracy: 99.69%

Classification Report
              precision    recall  f1-score   support

    Phishing       1.00      0.99      1.00     20116
  Legitimate       1.00      1.00      1.00     27043

    accuracy                           1.00     47159
   macro avg       1.00      1.00      1.00     47159
weighted avg       1.00      1.00      1.00     47159


Performance Summary
Precision:            0.9956
Recall:               0.9990
F1 Score:             0.9973
ROC-AUC:              0.9985
False Positive Rate:  0.59%
False Negative Rate:  0.10%

--- GroupKFold Cross Validation ---
Mean Accuracy: 99.72%
Std Dev: 0.02%

-

I originally thought that there was more data leakage problems, specifically I thought that XGBoost was just finding out the patterns from when the data was just being collected so I made it so that no domain in the test set ever appeared in the training phase. This is the model that I am going with for the project.